<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


<h1>Lab: Tokenization, Positional Encoding, and Attention Mechanisms in PyTorch</h1> 


**Estimated Time Needed**: 45 min


## Learning Objectives 

In this lab, you will learn to:

- Implement a tokenization pipeline using a Hugging Face AutoTokenizer 
- Compare subword outputs across at least two tokenizer models to observe vocabulary differences
- Apply sinusoidal positional encoding to tokenized input tensors and verify correct output dimensions after encoding
- Implement scaled dot-product attention using PyTorch matrix operations and compute and interpret the resulting attention score matrix
- Extend the single-head attention implementation to a multi-head attention module and verify output dimensions across query, key, and value projections


<h2>Table of Contents</h2>
<p>In this lab, you will use Hugging Face tokenizers, add positional encodings, and implement scaled dot-product and multi-head attention in PyTorch.</p>

<ol>
<li><a href="#Import-the-Libraries">Import the Libraries</a> 
</li>
<li><a href="#Build-the-Foundations-of-Transformer-Attention">Build the Foundations of Transformer Attention</a>     
</li>
<li><a href="#Load-Hugging-Face-AutoTokenizers">Load Hugging Face AutoTokenizers</a>     
</li>
<li><a href="#Compare-Subword-Tokenization">Compare Subword Tokenization</a>     
</li>
<li><a href="#Create-Tokenized-Input-Tensors">Create Tokenized Input Tensors</a>
</li>
<li><a href="#Apply-Sinusoidal-Positional-Encoding">Apply Sinusoidal Positional Encoding</a>
</li>
<li><a href="#Implement-Scaled-Dot-product-Self-attention">Implement Scaled Dot-product Self-attention</a>
</li> 
<li><a href="#Interpret-the-Attention-Score-Matrix">Interpret the Attention Score Matrix</a>
</li>    
<li><a href="#Implement-Multi-head-Attention">Implement Multi-head Attention</a>
</li>  

</li>   
</ol>



## Import the Libraries


You'll need the following libraries:


In [1]:
# Install PyTorch

%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Install Hugging Face Transformers and Python 3.7-compatible binary dependencies

%pip install --no-cache-dir --only-binary=:all: "transformers==4.30.2" "huggingface-hub==0.16.4" "tokenizers==0.13.3" "safetensors==0.3.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 148.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 260.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 129.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 203.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.6/761.6 kB 269.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import the libraries we need for this lab

import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer

torch.manual_seed(2)

## Build the Foundations of Transformer Attention


Transformer models use tokenization, positional information, and attention to represent text. Attention compares the query and key vectors to decide how strongly each token should use information from the value vectors.


<p style="text-align:center">
    <img src="https://commons.wikimedia.org/wiki/Special:Redirect/file/Self-Attention%20%28Scaled%20dot-product%20Attention%29.png" width="360" alt="Scaled dot-product self-attention diagram">
</p>
<p style="text-align:center"><em>Source: Wikimedia Commons, Self-Attention (Scaled dot-product Attention).png</em></p>


In this lab, you will build the core operations step by step so the tensor shapes and attention scores are visible.


## Load Hugging Face AutoTokenizers


Hugging Face <code>AutoTokenizer</code> loads the tokenizer configuration that matches a pretrained model. Different models can split the same word into different subword units.


In [4]:
# Load two tokenizer models

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")

Create sample text that includes words likely to be split into subwords:


In [5]:
# Create sample sentences

sentences = [
    "Transformers use attention to connect related tokens.",
    "Tokenization handles unfamiliar words like hyperparameterization.",
]

for sentence in sentences:
    print(sentence)

Transformers use attention to connect related tokens.
Tokenization handles unfamiliar words like hyperparameterization.


## Compare Subword Tokenization


Next, compare the subword output from the BERT tokenizer and the RoBERTa tokenizer:


In [6]:
# Compare subword tokens

sample_text = sentences[1]

bert_tokens = bert_tokenizer.tokenize(sample_text)
roberta_tokens = roberta_tokenizer.tokenize(sample_text)

print("BERT tokens:")
print(bert_tokens)
print("RoBERTa tokens:")
print(roberta_tokens)

BERT tokens:
['token', '##ization', 'handles', 'unfamiliar', 'words', 'like', 'hyper', '##para', '##meter', '##ization', '.']
RoBERTa tokens:
['Token', 'ization', 'Ġhandles', 'Ġunfamiliar', 'Ġwords', 'Ġlike', 'Ġhyper', 'param', 'eter', 'ization', '.']


Inspect the token IDs to see that each tokenizer uses its own vocabulary mapping:


In [7]:
# Compare token IDs

bert_ids = bert_tokenizer.convert_tokens_to_ids(bert_tokens)
roberta_ids = roberta_tokenizer.convert_tokens_to_ids(roberta_tokens)

print("BERT token IDs:")
print(bert_ids)
print("RoBERTa token IDs:")
print(roberta_ids)

BERT token IDs:
[19204, 3989, 16024, 16261, 2616, 2066, 23760, 28689, 22828, 3989, 1012]
RoBERTa token IDs:
[45643, 1938, 14617, 21942, 1617, 101, 8944, 46669, 5906, 1938, 4]


The token pieces and token IDs can differ because the models were trained with different tokenizer algorithms and vocabularies.


## Create Tokenized Input Tensors


Use one tokenizer to create padded input tensors. These tensors will be used to demonstrate positional encoding and attention.


In [8]:
# Tokenize text into tensors

encoded_batch = bert_tokenizer(
    sentences,
    padding=True,
    truncation=True,
    return_tensors="pt",
)

input_ids = encoded_batch["input_ids"]
attention_mask = encoded_batch["attention_mask"]

print("Input IDs shape:", input_ids.shape)
print("Attention mask shape:", attention_mask.shape)
print(input_ids)

Input IDs shape: torch.Size([2, 13])
Attention mask shape: torch.Size([2, 13])
tensor([[  101, 19081,  2224,  3086,  2000,  7532,  3141, 19204,  2015,  1012,
           102,     0,     0],
        [  101, 19204,  3989, 16024, 16261,  2616,  2066, 23760, 28689, 22828,
          3989,  1012,   102]])


Create token embeddings from the token IDs. In pretrained models, these embeddings are learned, but here you will use a small randomly initialized embedding layer.


In [9]:
# Create token embeddings

embedding_dim = 16
embedding_layer = nn.Embedding(bert_tokenizer.vocab_size, embedding_dim)
token_embeddings = embedding_layer(input_ids)

print("Token embeddings shape:", token_embeddings.shape)

Token embeddings shape: torch.Size([2, 13, 16])


## Apply Sinusoidal Positional Encoding


Attention does not process tokens in order by itself. Positional encoding adds position information to token embeddings while preserving the same tensor shape.


In [10]:
# Create sinusoidal positional encoding

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_length=100):
        super().__init__()
        position = torch.arange(max_length).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embedding_dim, 2) * (-math.log(10000.0) / embedding_dim))

        encoding = torch.zeros(max_length, embedding_dim)
        encoding[:, 0::2] = torch.sin(position * div_term)
        encoding[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("encoding", encoding.unsqueeze(0))

    def forward(self, token_embeddings):
        sequence_length = token_embeddings.size(1)
        return token_embeddings + self.encoding[:, :sequence_length, :]

Apply positional encoding and verify that the output dimensions match the input embedding dimensions:


In [11]:
# Apply positional encoding

positional_encoder = SinusoidalPositionalEncoding(embedding_dim=embedding_dim, max_length=50)
encoded_embeddings = positional_encoder(token_embeddings)

print("Before positional encoding:", token_embeddings.shape)
print("After positional encoding:", encoded_embeddings.shape)

Before positional encoding: torch.Size([2, 13, 16])
After positional encoding: torch.Size([2, 13, 16])


## Implement Scaled Dot-product Self-attention


Scaled dot-product attention computes similarity scores between queries and keys, normalizes them with softmax, and uses the result to combine value vectors.


In [12]:
# Implement scaled dot-product attention

def scaled_dot_product_attention(query, key, value, mask=None):
    key_dim = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) * (key_dim ** -0.5)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)
    return output, attention_weights, scores

Use the same encoded embeddings as query, key, and value to compute self-attention:


In [13]:
# Compute self-attention

mask = attention_mask.unsqueeze(1)
attention_output, attention_weights, raw_scores = scaled_dot_product_attention(
    encoded_embeddings,
    encoded_embeddings,
    encoded_embeddings,
    mask=mask,
)

print("Attention output shape:", attention_output.shape)
print("Attention weights shape:", attention_weights.shape)
print("Raw scores shape:", raw_scores.shape)

Attention output shape: torch.Size([2, 13, 16])
Attention weights shape: torch.Size([2, 13, 13])
Raw scores shape: torch.Size([2, 13, 13])


## Interpret the Attention Score Matrix


The attention score matrix has shape <code>[batch_size, sequence_length, sequence_length]</code>. Each row shows how much one query token attends to every key token.


In [14]:
# Inspect attention scores for the first sentence

tokens = bert_tokenizer.convert_ids_to_tokens(input_ids[0])
first_attention_matrix = attention_weights[0]

print(tokens)
print(first_attention_matrix)

['[CLS]', 'transformers', 'use', 'attention', 'to', 'connect', 'related', 'token', '##s', '.', '[SEP]', '[PAD]', '[PAD]']
tensor([[9.9793e-01, 3.4907e-05, 2.0110e-05, 1.5142e-04, 1.3720e-05, 1.1006e-04,
         8.7031e-05, 3.0735e-04, 1.9916e-04, 3.1112e-04, 8.3055e-04, 0.0000e+00,
         0.0000e+00],
        [2.0873e-03, 7.8261e-01, 3.8979e-02, 1.3933e-02, 1.1326e-02, 3.9678e-03,
         8.4394e-02, 4.5285e-02, 7.9312e-03, 5.2675e-03, 4.2228e-03, 0.0000e+00,
         0.0000e+00],
        [4.1581e-04, 1.3478e-02, 8.5231e-01, 2.3292e-02, 2.9712e-02, 4.7074e-02,
         8.7077e-03, 3.8953e-03, 3.7292e-03, 1.4334e-02, 3.0487e-03, 0.0000e+00,
         0.0000e+00],
        [2.0336e-02, 3.1293e-02, 1.5129e-01, 3.7553e-01, 5.5019e-03, 1.3712e-01,
         2.3877e-02, 8.8585e-02, 7.8055e-02, 4.5001e-02, 4.3407e-02, 0.0000e+00,
         0.0000e+00],
        [1.7171e-04, 2.3705e-03, 1.7985e-02, 5.1272e-04, 9.6919e-01, 3.6605e-03,
         2.7468e-03, 9.9128e-04, 2.1675e-04, 1.8896e-03, 2.65

Find the token with the highest attention weight for each query token:


In [15]:
# Interpret strongest attention links

for row_index, token in enumerate(tokens):
    strongest_index = int(first_attention_matrix[row_index].argmax().item())
    print(token, "attends most to", tokens[strongest_index])

[CLS] attends most to [CLS]
transformers attends most to transformers
use attends most to use
attention attends most to attention
to attends most to to
connect attends most to connect
related attends most to related
token attends most to token
##s attends most to ##s
. attends most to .
[SEP] attends most to [SEP]
[PAD] attends most to [CLS]
[PAD] attends most to [CLS]


Because the embeddings are randomly initialized in this lab, the attention links are useful for understanding mechanics, not linguistic meaning.


## Implement Multi-head Attention


Multi-head attention projects inputs into multiple query, key, and value spaces. Each head can learn a different attention pattern.


<p style="text-align:center">
    <img src="https://commons.wikimedia.org/wiki/Special:Redirect/file/Multiheaded%20attention%2C%20block%20diagram.png" width="650" alt="Multi-head attention block diagram">
</p>
<p style="text-align:center"><em>Source: Wikimedia Commons, Multiheaded attention, block diagram.png by dvgodoy</em></p>


In [16]:
# Implement multi-head attention

class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()
        if embedding_dim % num_heads != 0:
            raise ValueError("embedding_dim must be divisible by num_heads")

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.query_projection = nn.Linear(embedding_dim, embedding_dim)
        self.key_projection = nn.Linear(embedding_dim, embedding_dim)
        self.value_projection = nn.Linear(embedding_dim, embedding_dim)
        self.output_projection = nn.Linear(embedding_dim, embedding_dim)

    def split_heads(self, tensor):
        batch_size, sequence_length, embedding_dim = tensor.shape
        tensor = tensor.view(batch_size, sequence_length, self.num_heads, self.head_dim)
        return tensor.transpose(1, 2)

    def combine_heads(self, tensor):
        batch_size, num_heads, sequence_length, head_dim = tensor.shape
        tensor = tensor.transpose(1, 2).contiguous()
        return tensor.view(batch_size, sequence_length, num_heads * head_dim)

    def forward(self, query, key, value, mask=None):
        query = self.split_heads(self.query_projection(query))
        key = self.split_heads(self.key_projection(key))
        value = self.split_heads(self.value_projection(value))

        if mask is not None:
            mask = mask.unsqueeze(1)

        attention_output, attention_weights, scores = scaled_dot_product_attention(query, key, value, mask)
        combined_output = self.combine_heads(attention_output)
        output = self.output_projection(combined_output)
        return output, attention_weights, scores

Create a multi-head attention module and verify output dimensions:


In [17]:
# Verify multi-head attention dimensions

multi_head_attention = MultiHeadAttention(embedding_dim=embedding_dim, num_heads=4)
multi_head_output, multi_head_weights, multi_head_scores = multi_head_attention(
    encoded_embeddings,
    encoded_embeddings,
    encoded_embeddings,
    mask=attention_mask.unsqueeze(1),
)

print("Input shape:", encoded_embeddings.shape)
print("Output shape:", multi_head_output.shape)
print("Attention weights shape:", multi_head_weights.shape)
print("Attention scores shape:", multi_head_scores.shape)

Input shape: torch.Size([2, 13, 16])
Output shape: torch.Size([2, 13, 16])
Attention weights shape: torch.Size([2, 4, 13, 13])
Attention scores shape: torch.Size([2, 4, 13, 13])


The attention weights have shape <code>[batch_size, num_heads, sequence_length, sequence_length]</code>, giving one attention matrix per head.


<h2>About the Author:</h2> 

**Harish Pant** is an accomplished IT Consultant specializing in bridging the gap between complex AI technologies and practical business applications. Holding a Bachelor’s degree in Computer Science with a strong background in Machine Learning, Harish serves clients across diverse industries—including fintech, retail, and manufacturing—helping them navigate AI adoption. Harish is known for translating technical AI concepts into actionable business strategies and delivering engaging, high-impact presentations. 



<a href="https://dataplatform.cloud.ibm.com/registration/stepone?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork&context=cpdaas&apps=data_science_experience%2Cwatson_machine_learning"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork/Template/module%201/images/Watson_Studio.png"></a>


<!--
## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2026-05-11  | 1.0  | IBM Skills Network  |  Created lab for tokenization, positional encoding, and attention mechanisms in PyTorch |
-->


<hr>





## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>


<!--
## Change Log

| Date (YYYY-MM-DD) | Version | Changed By | Change Description |
| --- | --- | --- | --- |
| 2026-06-27 | 0.1 | Patsy Kravitz | Updated install cells for notebook kernel compatibility and Python 3.7 package stability. Replaced the unpinned Hugging Face Transformers install with pinned compatible versions for `transformers`, `huggingface-hub`, `tokenizers`, and `safetensors`; added `--only-binary=:all:` to prevent unsupported source-build dependency paths; updated the PyTorch install cell to use `%pip`; and removed three empty code cells from the notebook. |

-->
